# 04 · MLP multi-salida, DLinear y NLinear (implementación propia)

Modelos en `src/models/linear.py`, entrenados con `src/train.py` sobre el `WindowDataset` (L = 168, H = 24, stride 1 h, 3.25 M ventanas de entrenamiento).

| Modelo | Entrada | Idea |
|---|---|---|
| **NLinear** | solo y | resta el último valor, una capa lineal 168 → 24, lo suma de vuelta |
| **DLinear** | solo y | media móvil (k = 25) separa tendencia y resto; una capa lineal para cada parte |
| **MLP** | y + calendario de las 168 h de entrada + calendario de las 24 h del horizonte | 2 capas ocultas con GELU y dropout, salida densa de 24 |

Los tres son **multi-salida** (devuelven los 24 pasos de una vez) y **globales** (un modelo para los 150 clientes).

**Protocolo:**
1. Cada configuración se entrena con objetivos hasta 2014-06-30 y early stopping sobre validación (jul–sep, orígenes a las 00:00).
2. Se elige la mejor por MASE de validación.
3. Se reentrena una vez con datos hasta 2014-09-30 durante las épocas que eligió el early stopping.
4. Se evalúa una sola vez en prueba con `backtest()`.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import matplotlib.pyplot as plt
from src import data as D, metrics as M, backtest as B, train as T
from src.dataset import WindowDataset
from src.models.linear import MLP, DLinear, NLinear

device = "cuda" if torch.cuda.is_available() else "cpu"
panel = D.build_panel()
scale = M.mase_scale(panel)
scaler = D.SeriesScaler().fit(panel)       # z-score por serie, estadísticos de entrenamiento
z = scaler.transform(panel)

L, H = 168, 24
train_ds = WindowDataset(z, L=L, H=H, split="train", with_future=True)
val_ds = WindowDataset(z, L=L, H=H, split="val", origin_hour=0, with_future=True)
full_ds = WindowDataset(z, L=L, H=H, split=None, target_end=D.VAL_END, with_future=True)
print(f"train {len(train_ds):,} · val {len(val_ds):,} · train+val {len(full_ds):,} ventanas · {device}")

## 1. Iteraciones en validación

Para cada modelo se prueba la normalización (solo z-score por serie contra z-score + **RevIN**, que además normaliza cada ventana) y, en el MLP, el tamaño y la profundidad. Cada iteración guarda su historial por epoch, parámetros, tiempos y memoria en `outputs/runs/<nombre>/`.

In [ ]:
n_feat, n_fut = train_ds.n_features, train_ds.n_future_features
iteraciones = {
    "NLinear":        lambda: NLinear(L, H),
    "NLinear_revin":  lambda: NLinear(L, H, revin=True),
    "DLinear":        lambda: DLinear(L, H),
    "DLinear_revin":  lambda: DLinear(L, H, revin=True),
    "MLP_512x2":      lambda: MLP(L, H, n_feat, n_fut, hidden=512, layers=2, dropout=0.1),
    "MLP_512x2_revin":lambda: MLP(L, H, n_feat, n_fut, hidden=512, layers=2, dropout=0.1, revin=True),
    "MLP_1024x3":     lambda: MLP(L, H, n_feat, n_fut, hidden=1024, layers=3, dropout=0.2),
}
lrs = {"NLinear": 1e-3, "DLinear": 1e-3, "MLP": 5e-4}

resultados = {}
for name, make in iteraciones.items():
    fam = name.split("_")[0]
    res = T.fit(make(), train_ds, val_ds, scaler, scale, name=name, epochs=15, patience=3,
                lr=lrs[fam], batch_size=1024, device=device, verbose=False,
                config={"familia": fam, "normalizacion": "zscore+RevIN" if "revin" in name else "zscore por serie",
                        "meses_historia": 30})
    resultados[name] = res
    i = res.info
    print(f"{name:16s} MASE val {i['val_MASE']:.4f}  MAE {i['val_MAE']:.2f}  epochs {i['epochs_corridos']:2d} "
          f"(mejor {i['mejor_epoch']:2d})  {i['train_s_total']:.0f}s  params {i['params_total']:,}")

In [ ]:
cols = ["familia", "normalizacion", "L", "H", "meses_historia", "lr", "batch_size", "params_total", "params_entrenables",
        "epochs_corridos", "mejor_epoch", "val_MAE", "val_MASE", "train_s_por_epoch", "train_s_total", "gpu_mem_pico_MB"]
iter_tab = pd.DataFrame({k: r.info for k, r in resultados.items()}).T[cols]
iter_tab

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=False)
for ax, fam in zip(axes, ["NLinear", "DLinear", "MLP"]):
    for name, r in resultados.items():
        if not name.startswith(fam):
            continue
        h = r.history
        line, = ax.plot(h["epoch"], h["train_loss"], ls="--", alpha=.7)
        ax.plot(h["epoch"], h["val_loss"], color=line.get_color(), label=name)
    ax.set_title(f"{fam}: pérdida (-- train, — val)"); ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[0].set_ylabel("MSE (escala normalizada)")
fig.tight_layout()
fig.savefig(D.ROOT / "outputs" / "figs" / "curvas_lineales.png", dpi=150)

## 2. Modelo final de cada familia y evaluación en prueba

In [ ]:
filas = []
for fam in ["NLinear", "DLinear", "MLP"]:
    cand = {k: r for k, r in resultados.items() if k.startswith(fam)}
    best = min(cand, key=lambda k: cand[k].info["val_MASE"])
    info = cand[best].info
    print(f"{fam}: mejor configuración {best} (MASE val {info['val_MASE']:.4f}, {info['mejor_epoch']} epochs)")
    model, rinfo = T.refit(iteraciones[best](), full_ds, epochs=info["mejor_epoch"], name=best,
                           lr=lrs[fam], batch_size=1024, device=device, verbose=False)
    fn = B.torch_forecast_fn(model, scaler, L=L, H=H, with_future=True, device=device)
    fc = B.backtest(fn, panel, name=fam)
    row, _ = B.finish(fam, fc, panel, scale, familia="MLP y lineales", L=L, estrategia="multi-salida",
                      covariables=(fam == "MLP"), config=best, normalizacion=info["normalizacion"],
                      params_total=rinfo["params_total"], params_entrenables=rinfo["params_entrenables"],
                      train_s=info["train_s_total"] + rinfo["refit_s"], val_MASE=info["val_MASE"],
                      gpu_mem_pico_MB=max(info["gpu_mem_pico_MB"], rinfo["gpu_mem_pico_MB_refit"]))
    filas.append(row)
    torch.save(model.state_dict(), D.ROOT / "outputs" / "runs" / best / "modelo_final.pt")

tabla = pd.DataFrame(filas).set_index("modelo")
tabla[["config", "params_total", "val_MASE", "MAE", "RMSE", "WAPE", "MASE", "MASE_navidad", "train_s", "inference_s"]]

**Preguntas para el informe** (se responden con estas tablas):
- ¿RevIN ayudó? Comparar `X` contra `X_revin` en MASE de validación.
- ¿Una pérdida de entrenamiento más baja implica siempre un mejor MASE en validación? Comparar las curvas: el MLP grande baja más la pérdida de entrenamiento que los lineales, y hay que ver si eso se traduce en validación.